In [1]:
!git clone https://github.com/guyernest/advanced-rag.git
%cd advanced-rag
!pip install -q -r requirements.txt

fatal: destination path 'advanced-rag' already exists and is not an empty directory.
/kaggle/working/advanced-rag
^C
ERROR: Operation cancelled by user


In [2]:
%cd advanced-rag

[Errno 2] No such file or directory: 'advanced-rag'
/kaggle/working/advanced-rag


# Simple RAG Implementation

Based on [Alfredo Deza's GitHub Repository](https://github.com/alfredodeza/learn-retrieval-augmented-generation).

In this notebook we will build a simple RAG application based on a structured CSV file with wine rating. We will:
* [Load the small dataset](#loading-the-dataset).
* [Encode a column using vector embedding](#Encode-using-vector-embedding).
* [**R**etrieve some of the rows based on a query using semantic similarity](#retrieve-sematically-relevant-data-based-on-users-query).
* [**A**ugment the prompt to the LLM with the retrieved data](#augment-the-prompt-to-the-llm-with-retrieved-data).
* [**G**enerate a reply to the user's query based on the retrieved rows](#generate-reply-to-the-users-query).

### Visual improvements

We will use [rich library](https://github.com/Textualize/rich), and `rich-theme-manager` to make the output more readable, and supress warning messages.

In [3]:
from rich.console import Console
from rich.style import Style
import pathlib
from rich_theme_manager import Theme, ThemeManager

THEMES = [
    Theme(
        name="dark",
        description="Dark mode theme",
        tags=["dark"],
        styles={
            "repr.own": Style(color="#e87d3e", bold=True),      # Class names
            "repr.tag_name": "dim cyan",                        # Adjust tag names
            "repr.call": "bright_yellow",                       # Function calls and other symbols
            "repr.str": "bright_green",                         # String representation
            "repr.number": "bright_red",                        # Numbers
            "repr.none": "dim white",                           # None
            "repr.attrib_name": Style(color="#e87d3e", bold=True),    # Attribute names
            "repr.attrib_value": "bright_blue",                 # Attribute values
            "default": "bright_white on black"                  # Default text and background
        },
    ),
    Theme(
        name="light",
        description="Light mode theme",
        styles={
            "repr.own": Style(color="#22863a", bold=True),          # Class names
            "repr.tag_name": Style(color="#00bfff", bold=True),     # Adjust tag names
            "repr.call": Style(color="#ffff00", bold=True),         # Function calls and other symbols
            "repr.str": Style(color="#008080", bold=True),          # String representation
            "repr.number": Style(color="#ff6347", bold=True),       # Numbers
            "repr.none": Style(color="#808080", bold=True),         # None
            "repr.attrib_name": Style(color="#ffff00", bold=True),  # Attribute names
            "repr.attrib_value": Style(color="#008080", bold=True), # Attribute values
            "default": Style(color="#000000", bgcolor="#ffffff"),   # Default text and background
        },
    ),
]

theme_dir = pathlib.Path("themes").expanduser()
theme_dir.expanduser().mkdir(parents=True, exist_ok=True)

theme_manager = ThemeManager(theme_dir=theme_dir, themes=THEMES)
theme_manager.list_themes()

dark = theme_manager.get("dark")
theme_manager.preview_theme(dark)

 Theme  Description       Tags  Path               
 dark   Dark mode theme   dark  themes/dark.theme  
 light  Light mode theme        themes/light.theme

                                      Theme: dark - themes/dark.theme                                      
┌───────────────────┬───────────────┬───────┬─────────┬─────────┬────────────────┬────────────────────────┐
│ style             │ color         │ color │ bgcolor │ bgcolor │ attributes     │ example                │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ default           │ bright_white  │ █████ │ black   │ █████   │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.attrib_name  │ #e87d3e       │ █████ │ None    │         │ b------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.attrib_value │ bright_blue   │ █████ │ None    │         │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.call         │ bright_yellow │ █████ │ None    │         │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.none         │ white         │ █████ │ None    │         │ -d------------ │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.number       │ bright_red    │ █████ │ None    │         │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.own          │ #e87d3e       │ █████ │ None    │         │ b------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.str          │ bright_green  │ █████ │ None    │         │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.tag_name     │ cyan          │ █████ │ None    │         │ -d------------ │ The quick brown fox... │
└───────────────────┴───────────────┴───────┴─────────┴─────────┴────────────────┴────────────────────────┘
┌─ attributes legend ──────────────────────────────────────────────────────────────────┐
│  b: bold, d: dim, i: italic, u: underline, U: double underline, B: blink, 2: blink2  │
│  r: reverse, c: conceal, s: strike, f: frame, e: encircle, o: overline, L: Link      │
└──────────────────────────────────────────────────────────────────────────────────────┘

In [4]:
from rich.console import Console

dark = theme_manager.get("dark")
# Create a console with the dark theme
console = Console(theme=dark)


In [5]:
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

## Loading the Dataset

Since the data is in a simple, small and structured CSV file, we can load it using Pandas.

In [6]:
import pandas as pd

data = (
    pd
    .read_csv('data/top_rated_wines.csv')
    # .query('variety.notna()')
    # .reset_index(drop=True)
    # .to_dict('records')
)
console.print(data[:2])

name  \
0    3 Rings Reserve Shiraz 2004   
1  Abreu Vineyards Cappella 2007   

                                              region   variety  rating  \
0  Barossa Valley, Barossa, South Australia, Aust...  Red Wine    96.0   
1                            Napa Valley, California  Red Wine    96.0   

                                               notes  
0  Vintage Comments : Classic Barossa vintage con...  
1  Cappella is a proprietary blend of two clones ...

In [7]:
import pandas as pd

df = pd.read_csv("/kaggle/input/nist-800-53/800-53-rev4-controls.csv")

df["notes"] = (
    "Control Family: " + df["FAMILY"].fillna('') + ". " +
    "Control ID: " + df["NAME"].fillna('') + ". " +
    "Title: " + df["TITLE"].fillna('') + ". " +
    "Description: " + df["DESCRIPTION"].fillna('') + ". " +
    "Supplemental Guidance: " + df["SUPPLEMENTAL GUIDANCE"].fillna('') + ". " +
    "Related Controls: " + df["RELATED"].fillna('') + ". "
)

data = df.to_dict(orient="records")


console.print(data[:2])

[
    {
        'FAMILY': 'ACCESS CONTROL',
        'NAME': 'AC-1',
        'TITLE': 'ACCESS CONTROL POLICY AND PROCEDURES',
        'PRIORITY': 'P1',
        'BASELINE-IMPACT': 'LOW,MODERATE,HIGH',
        'DESCRIPTION': 'The organization:',
        'SUPPLEMENTAL GUIDANCE': 'This control addresses the establishment of policy and procedures for the 
effective implementation of selected security controls and control enhancements in the AC family. Policy and 
procedures reflect applicable federal laws, Executive Orders, directives, regulations, policies, standards, and 
guidance. Security program policies and procedures at the organization level may make the need for system-specific 
policies and procedures unnecessary. The policy can be included as part of the general information security policy 
for organizations or conversely, can be represented by multiple policies reflecting the complex nature of certain 
organizations. The procedures can be established for the security program in general and for particular information
systems, if needed. The organizational risk management strategy is a key factor in establishing policy and 
procedures.',
        'RELATED': 'PM-9',
        'notes': 'Control Family: ACCESS CONTROL. Control ID: AC-1. Title: ACCESS CONTROL POLICY AND PROCEDURES. 
Description: The organization:. Supplemental Guidance: This control addresses the establishment of policy and 
procedures for the effective implementation of selected security controls and control enhancements in the AC 
family. Policy and procedures reflect applicable federal laws, Executive Orders, directives, regulations, policies,
standards, and guidance. Security program policies and procedures at the organization level may make the need for 
system-specific policies and procedures unnecessary. The policy can be included as part of the general information 
security policy for organizations or conversely, can be represented by multiple policies reflecting the complex 
nature of certain organizations. The procedures can be established for the security program in general and for 
particular information systems, if needed. The organizational risk management strategy is a key factor in 
establishing policy and procedures.. Related Controls: PM-9. '
    },
    {
        'FAMILY': nan,
        'NAME': 'AC-1a.',
        'TITLE': nan,
        'PRIORITY': nan,
        'BASELINE-IMPACT': nan,
        'DESCRIPTION': 'Develops, documents, and disseminates to [Assignment: organization-defined personnel or 
roles]:',
        'SUPPLEMENTAL GUIDANCE': nan,
        'RELATED': nan,
        'notes': 'Control Family: . Control ID: AC-1a.. Title: . Description: Develops, documents, and disseminates
to [Assignment: organization-defined personnel or roles]:. Supplemental Guidance: . Related Controls: . '
    }
]

## Encode using Vector Embedding

We will use one of the popular open source vector databases, [Qdrant](https://qdrant.tech/), and one of the popular embedding encoder and text transformer libraries, [SentenceTransformer](https://sbert.net/).

In [8]:
!pip install --upgrade torch torchvision

In [9]:
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

# create the vector database client
qdrant = QdrantClient(":memory:") # Create in-memory Qdrant instance

# Create the embedding encoder
encoder = SentenceTransformer('all-MiniLM-L6-v2') # Model to create embeddings

2025-12-09 08:59:49.909751: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765270789.932955     619 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765270789.940947     619 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [10]:
# Create collection to store the wine rating data
# collection_name="top_wines"

collection_name = "nist_code"

qdrant.recreate_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
        distance=models.Distance.COSINE
    )
)

True

### Loading the data into the vector database

We will use the (vector) collection that we created above, to go over all the `notes` column of the wine dataset, and encode it into embedding vector, and store it in the vector database. The indexing of the data to allow quick retrieval is running in the background as we load it.

This step will take a few seconds (less than a minute on my laptop).

In [11]:
import pandas as pd
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

# Assume qdrant and encoder are already defined from previous cells
# create the vector database client - This should ideally be done once
# qdrant = QdrantClient(":memory:")

# Create the embedding encoder - This should ideally be done once
# encoder = SentenceTransformer('all-MiniLM-L6-v2')

#collection_name="top_wines"

collection_name="nist_code"

# Create collection to store the wine rating data if it doesn't exist
if not qdrant.collection_exists(collection_name=collection_name):
    qdrant.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
            distance=models.Distance.COSINE
        )
    )


# vectorize!
qdrant.upload_points(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=idx,
            vector=encoder.encode(doc["notes"]).tolist(),
            payload=doc
        ) for idx, doc in enumerate(data) # data is the variable holding all the wines
    ]
)

In [12]:
console.print(qdrant.get_collection(collection_name=collection_name))

CollectionInfo(
    status=<CollectionStatus.GREEN: 'green'>,
    optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>,
    vectors_count=None,
    indexed_vectors_count=0,
    points_count=1682,
    segments_count=1,
    config=CollectionConfig(
        params=CollectionParams(
            vectors=VectorParams(
                size=384,
                distance=<Distance.COSINE: 'Cosine'>,
                hnsw_config=None,
                quantization_config=None,
                on_disk=None,
                datatype=None,
                multivector_config=None
            ),
            shard_number=None,
            sharding_method=None,
            replication_factor=None,
            write_consistency_factor=None,
            read_fan_out_factor=None,
            on_disk_payload=None,
            sparse_vectors=None
        ),
        hnsw_config=HnswConfig(
            m=16,
            ef_construct=100,
            full_scan_threshold=10000,
            max_indexing_threads=0,
            on_disk=None,
            payload_m=None
        ),
        optimizer_config=OptimizersConfig(
            deleted_threshold=0.2,
            vacuum_min_vector_number=1000,
            default_segment_number=0,
            max_segment_size=None,
            memmap_threshold=None,
            indexing_threshold=20000,
            flush_interval_sec=5,
            max_optimization_threads=1
        ),
        wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0),
        quantization_config=None
    ),
    payload_schema={}
)

## **R**etrieve sematically relevant data based on user's query

Once the data is loaded into the vector database and the indexing process is done, we can start using our simple RAG system.

In [13]:
user_prompt = "My small business shares passwords, please inform me of the vulnerability and provide a NIST control code for the vulnerability"

### Encoding the user's query

We will use the same encoder that we used to encode the document data to encode the query of the user.
This way we can search results based on semantic similarity.

In [14]:
query_vector = encoder.encode(user_prompt).tolist()

### Search similar rows

We can now take the embedding encoding of the user's query and use it to find similar rows in the vector database.

# **Wine Retrieval**

In [24]:
# Search time for awesome wines!

hits = qdrant.search(
    collection_name=collection_name,
    query_vector=query_vector,
    limit=3
)

In [25]:
from rich.console import Console
from rich.text import Text
from rich.table import Table

table = Table(title="Retrieval Results", show_lines=True)

table.add_column("Name", style="#e0e0e0")
table.add_column("Region", style="bright_red")
table.add_column("Variety", style="green")
table.add_column("Rating", style="yellow")
table.add_column("Notes", style="#89ddff")
table.add_column("Score", style="#a6accd")

for hit in hits:
    table.add_row(
        hit.payload["name"],
        hit.payload["region"],
        hit.payload["variety"],
        str(hit.payload["rating"]),
        f'{hit.payload["notes"][:50]}...',
        f"{hit.score:.4f}"
    )

console.print(table)

KeyError: 'name'

# **NIST Retreival**

In [15]:
# Search time for awesome codes!

hits = qdrant.search(
    collection_name=collection_name,
    query_vector=query_vector,
    limit=3
)

In [16]:
from rich.console import Console
from rich.table import Table

console = Console()

table = Table(title="NIST 800-53 Retrieval Results", show_lines=True)

table.add_column("Control ID", style="bright_red")
table.add_column("Title", style="green")
table.add_column("Family", style="yellow")
table.add_column("Description", style="#89ddff")
table.add_column("Score", style="#a6accd")

for hit in hits:
    payload = hit.payload or {}

    control_id = str(payload.get("NAME", "N/A"))
    title      = str(payload.get("TITLE", "N/A"))
    family     = str(payload.get("FAMILY", "N/A"))

    desc_raw = payload.get("DESCRIPTION", "")
    desc = "" if desc_raw is None else str(desc_raw)
    if len(desc) > 80:
        desc = desc[:80] + "..."

    score = f"{hit.score:.4f}"

    table.add_row(control_id, title, family, desc, score)

console.print(table)


                                           NIST 800-53 Retrieval Results                                           
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Control ID   ┃ Title                       ┃ Family                      ┃ Description                 ┃ Score  ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ SA-15 (7)(a) │ nan                         │ nan                         │ Perform an automated        │ 0.4814 │
│              │                             │                             │ vulnerability analysis      │        │
│              │                             │                             │ using [Assignment:          │        │
│              │                             │                             │ organization-defi...        │        │
├──────────────┼─────────────────────────────┼─────────────────────────────┼─────────────────────────────┼────────┤
│ SA-15 (8)    │ REUSE OF THREAT /           │ SYSTEM AND SERVICES         │ The organization requires   │ 0.4749 │
│              │ VULNERABILITY INFORMATION   │ ACQUISITION                 │ the developer of the        │        │
│              │                             │                             │ information system, system  │        │
│              │                             │                             │ compon...                   │        │
├──────────────┼─────────────────────────────┼─────────────────────────────┼─────────────────────────────┼────────┤
│ IA-5 (1)(b)  │ nan                         │ nan                         │ Enforces at least the       │ 0.4555 │
│              │                             │                             │ following number of changed │        │
│              │                             │                             │ characters when new         │        │
│              │                             │                             │ passwords ...               │        │
└──────────────┴─────────────────────────────┴─────────────────────────────┴─────────────────────────────┴────────┘

## **A**ugment the prompt to the LLM with retrieved data

In our simple example, we will simply take the top 3 results and use them as is in the prompt to the generation LLM.

## **G**enerate reply to the user's query

We will use one of the most popular generative AI LLMs from [OpenAI](https://platform.openai.com/docs/models).

In [30]:
from dotenv import load_dotenv

load_dotenv()

False

### First let's try without **R**etrieval

We can ask the LLM to recommend based only on the user prompt.

In [17]:
!pip install -U transformers accelerate torchvision --quiet


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [18]:
!pip install -U torch --index-url https://download.pytorch.org/whl/cu121


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Looking in indexes: https://download.pytorch.org/whl/cu121


In [19]:
import transformers, torch
print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)


Transformers: 4.57.3
Torch: 2.9.1+cu128


# **Test for NIST Control Codes**

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from rich.panel import Panel

# Select GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

# Chat messages
messages=[
        {"role": "system", "content": "You are chatbot, a cybersecurity specialist. Your top priority is to help the user answer questions about his small business"},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": "Here is my cybersecurity recommendation:"}
]

# Tokenize and move to GPU
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(device)

# Generate output
outputs = model.generate(
    **inputs,
    max_new_tokens=400,
)

# Decode only the generated portion
generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

styled_panel = Panel(
    generated,
    title="Wine Recommendation without Retrieval",
    expand=False,
    border_style="bold green",
    padding=(1, 1)
)

console.print(styled_panel)

print(generated)

Using device: cuda


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

In [ ]:
# define a variable to hold the search results
search_results = [hit.payload for hit in hits]

In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

# Chat messages
messages=[
        {"role": "system", "content": "You are chatbot, a wine specialist. Your top priority is to help guide users into selecting amazing wine and guide them with their requests."},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": str(search_results)}
]

# Tokenize and move to GPU
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(device)

# Generate output
outputs = model.generate(
    **inputs,
    max_new_tokens=1000,
)

# Decode only the generated portion
generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

styled_panel = Panel(
    generated,
    title="Wine Recommendation with Retrieval",
    expand=False,
    border_style="bold green",
    padding=(1, 1)
)

console.print(styled_panel)
print(generated)

# **Wine Test**

In [20]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from rich.panel import Panel

# Select GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

# Chat messages
messages=[
        {"role": "system", "content": "You are chatbot, a wine specialist. Your top priority is to help guide users into selecting amazing wine and guide them with their requests."},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": "Here is my wine recommendation:"}
]

# Tokenize and move to GPU
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(device)

# Generate output
outputs = model.generate(
    **inputs,
    max_new_tokens=400,
)

# Decode only the generated portion
generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

styled_panel = Panel(
    generated,
    title="Wine Recommendation without Retrieval",
    expand=False,
    border_style="bold green",
    padding=(1, 1)
)

console.print(styled_panel)

print(generated)

Using device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

╭───────────────────────────────────── Wine Recommendation without Retrieval ─────────────────────────────────────╮
│                                                                                                                 │
│ For an amazing Malbec from Argentina, I would highly recommend the **"Vidal Finísica" Malbec"** from Mendoza.   │
│ This wine is known for its rich, dark fruit flavors and smooth tannins. It's often praised for its balance      │
│ between complexity and approachability.                                                                         │
│                                                                                                                 │
│ **Vidal Finísica** is part of the Vidal family, which is one of the most widely planted grape varieties in      │
│ Argentina due to its resistance to frost. The "Finísica" designation indicates that it comes from a vineyard in │
│ the Finísima sub-zone of Luján de Cuyo, known for its cool climate which can produce more elegant Malbecs.      │
│                                                                                                                 │
│ If you're looking for something slightly different or if you're seeking reviews or specific tasting notes, I    │
│ can certainly provide more detailed information about this wine!                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

For an amazing Malbec from Argentina, I would highly recommend the **"Vidal Finísica" Malbec"** from Mendoza. This wine is known for its rich, dark fruit flavors and smooth tannins. It's often praised for its balance between complexity and approachability.

**Vidal Finísica** is part of the Vidal family, which is one of the most widely planted grape varieties in Argentina due to its resistance to frost. The "Finísica" designation indicates that it comes from a vineyard in the Finísima sub-zone of Luján de Cuyo, known for its cool climate which can produce more elegant Malbecs.

If you're looking for something slightly different or if you're seeking reviews or specific tasting notes, I can certainly provide more detailed information about this wine!


### Now, add **R**etrieval Results

The recommendation sounds great, however, we don't have this wine in our inventory and menu. Moreover, new wines may be newly available that were not part of the pre-training of the LLM.

We will run the same query with the **R**trieval results and get better recommendations for our business needs.

In [21]:
# define a variable to hold the search results
search_results = [hit.payload for hit in hits]

In [24]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

# Chat messages
messages=[
        {"role": "system", "content": "You are chatbot, a wine specialist. Your top priority is to help guide users into selecting amazing wine and guide them with their requests."},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": str(search_results)}
]

# Tokenize and move to GPU
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(device)

# Generate output
outputs = model.generate(
    **inputs,
    max_new_tokens=1000,
)

# Decode only the generated portion
generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

styled_panel = Panel(
    generated,
    title="Wine Recommendation with Retrieval",
    expand=False,
    border_style="bold green",
    padding=(1, 1)
)

console.print(styled_panel)
print(generated)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

╭────────────────────────────────────── Wine Recommendation with Retrieval ───────────────────────────────────────╮
│                                                                                                                 │
│ Based on the recommendations provided, the **Catena Zapata Argentino Vineyard Malbec 2004** stands out as an    │
│ incredible choice. This wine is highly rated at 98/100 and is known for its remarkable structure and            │
│ complexity. It's aged for 17 months in new French oak, offering a rich tapestry of flavors including wood       │
│ smoke, creosote, pepper, clove, black cherry, and blackberry.                                                   │
│                                                                                                                 │
│ Given your interest in Malbec and the quality of this vintage, I would recommend exploring this exceptional     │
│ wine. It's particularly noted for its elegance and longevity, making it a fantastic investment if you're        │
│ looking to enjoy it over the next couple of decades. Enjoy your tasting!                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Based on the recommendations provided, the **Catena Zapata Argentino Vineyard Malbec 2004** stands out as an incredible choice. This wine is highly rated at 98/100 and is known for its remarkable structure and complexity. It's aged for 17 months in new French oak, offering a rich tapestry of flavors including wood smoke, creosote, pepper, clove, black cherry, and blackberry.

Given your interest in Malbec and the quality of this vintage, I would recommend exploring this exceptional wine. It's particularly noted for its elegance and longevity, making it a fantastic investment if you're looking to enjoy it over the next couple of decades. Enjoy your tasting!
